# Core-score sensitivity and robustness scaffold

## 1. Purpose and scope

This notebook is a validation-oriented scaffold for consolidating core-score sensitivity and robustness checks that are currently spread across:

- `notebooks/04_directed_results_top30.ipynb`
- `notebooks/04_directed_results_plus3.ipynb`
- `notebooks/overlap_confirmation.ipynb`
- robustness sections in `notebooks/04_directed_results.ipynb`

For this initial scaffold, the notebook intentionally avoids full recomputation, scientific calculation changes, manuscript figure generation, and duplicated analysis blocks. Future work should use this notebook to validate or regenerate namespaced sensitivity artifacts without replacing the canonical manuscript workflow.

## 2. Inputs and artifact policy

Planned inputs should remain read-only unless a future implementation explicitly adds a controlled regeneration step. The notebook may inspect existing project artifacts, including prior sensitivity files and canonical core-score inputs, but it must not alter source data or manuscript outputs.

Artifact policy for this notebook:

- Read from existing notebooks and results only as needed for validation.
- Do not write to `results/figures/paper/`.
- Do not create canonical manuscript outputs.
- Do not create DOCX manuscript tables.
- Keep any future generated files under `results/sensitivity/` with variant-specific subdirectories.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SENSITIVITY_DIR = PROJECT_ROOT / "results" / "sensitivity"

SOURCE_NOTEBOOKS = {
    "canonical": PROJECT_ROOT / "notebooks" / "04_directed_results.ipynb",
    "top30": PROJECT_ROOT / "notebooks" / "04_directed_results_top30.ipynb",
    "strict_plus3": PROJECT_ROOT / "notebooks" / "04_directed_results_plus3.ipynb",
    "overlap_confirmation": PROJECT_ROOT / "notebooks" / "overlap_confirmation.ipynb",
}

EXPECTED_SENSITIVITY_ARTIFACTS = {
    "top30_scores": SENSITIVITY_DIR / "top30" / "core_scores_top30.csv",
    "top100_scores": SENSITIVITY_DIR / "top100" / "core_scores_top100.csv",
    "sensitivity_readme": SENSITIVITY_DIR / "README.md",
}

PROJECT_ROOT, SENSITIVITY_DIR

## 3. Sensitivity variants

Initial variant registry:

| variant | top_n | min_votes | target_up | target_dn | notes |
|---|---:|---:|---:|---:|---|
| `canonical_top50` | 50 | 2 | 42 | 35 | Canonical reference settings used for comparison. |
| `top30` | 30 | 2 | 42 | 35 | Reduced ranked-gene threshold sensitivity check. |
| `top100` | 100 | 2 | 42 | 35 | Expanded ranked-gene threshold sensitivity check. |
| `strict_plus3` | 50 | 3 | 18 | 6 | Stricter voting threshold sensitivity check. |

The registry below is metadata only. It should be used by future validation code to avoid hard-coding variant parameters in multiple places.

In [ ]:
VARIANTS = {
    "canonical_top50": {
        "top_n": 50,
        "min_votes": 2,
        "target_up": 42,
        "target_dn": 35,
    },
    "top30": {
        "top_n": 30,
        "min_votes": 2,
        "target_up": 42,
        "target_dn": 35,
    },
    "top100": {
        "top_n": 100,
        "min_votes": 2,
        "target_up": 42,
        "target_dn": 35,
    },
    "strict_plus3": {
        "top_n": 50,
        "min_votes": 3,
        "target_up": 18,
        "target_dn": 6,
    },
}

VARIANTS

## 4. Planned validation checks

Future implementation should add lightweight validation checks before any regeneration logic. Candidate checks:

1. Confirm required source notebooks and input artifacts exist.
2. Confirm all variant parameters match the registry above.
3. Confirm sensitivity artifacts are written only under `results/sensitivity/`.
4. Compare regenerated sensitivity tables with existing artifacts when present.
5. Validate expected columns, row counts, target up/down counts, and stable sort keys.
6. Confirm no canonical manuscript figure or DOCX outputs are created by this notebook.
7. Summarize pass/fail status in notebook output without changing scientific calculations.

In [ ]:
artifact_status = {
    "source_notebooks": {
        name: path.exists() for name, path in SOURCE_NOTEBOOKS.items()
    },
    "expected_sensitivity_artifacts": {
        name: path.exists() for name, path in EXPECTED_SENSITIVITY_ARTIFACTS.items()
    },
}

artifact_status

## 4A. Historical and namespaced artifact inventory

Some sensitivity artifacts already have namespaced replacements under `results/sensitivity/`, including the `top30` and `top100` core-score CSV files. Other artifacts remain historical-only notebook-root files that are retained as fallback or provenance references.

This notebook currently performs validation only. The inventory below records preferred namespaced paths where available and historical fallback paths where artifacts have not yet been migrated. The checks are read-only and do not recompute core scores, regenerate sensitivity artifacts, create figures, or rewrite historical files.

In [ ]:
ARTIFACT_REGISTRY = {
    "top30_scores": {
        "preferred_path": SENSITIVITY_DIR / "top30" / "core_scores_top30.csv",
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_scores_top30.csv",
        "artifact_type": "csv",
        "status_role": "namespaced_preferred_with_historical_fallback",
    },
    "top50_scores": {
        "preferred_path": None,
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_scores_top50.csv",
        "artifact_type": "csv",
        "status_role": "historical_only_reference",
    },
    "top100_scores": {
        "preferred_path": SENSITIVITY_DIR / "top100" / "core_scores_top100.csv",
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_scores_top100.csv",
        "artifact_type": "csv",
        "status_role": "namespaced_preferred_with_historical_fallback",
    },
    "core_v2_pickle": {
        "preferred_path": None,
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_v2.pkl",
        "artifact_type": "pickle",
        "status_role": "historical_only_provenance",
    },
    "core_v3_pickle": {
        "preferred_path": None,
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_v3.pkl",
        "artifact_type": "pickle",
        "status_role": "historical_only_provenance",
    },
    "core_v2_top30_pickle": {
        "preferred_path": None,
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_v2_top30.pkl",
        "artifact_type": "pickle",
        "status_role": "historical_only_top30_provenance",
    },
    "core_v2_top100_pickle": {
        "preferred_path": None,
        "fallback_path": PROJECT_ROOT / "notebooks" / "core_v2_top100.pkl",
        "artifact_type": "pickle",
        "status_role": "historical_only_top100_provenance",
    },
}

ARTIFACT_REGISTRY

In [ ]:
import pandas as pd

artifact_inventory_rows = []
for artifact_name, metadata in ARTIFACT_REGISTRY.items():
    preferred_path = metadata["preferred_path"]
    fallback_path = metadata["fallback_path"]
    preferred_exists = preferred_path.exists() if preferred_path is not None else False
    fallback_exists = fallback_path.exists() if fallback_path is not None else False
    selected_path = preferred_path if preferred_exists else fallback_path if fallback_exists else preferred_path or fallback_path

    artifact_inventory_rows.append(
        {
            "artifact": artifact_name,
            "preferred_exists": preferred_exists,
            "fallback_exists": fallback_exists,
            "selected_path": str(selected_path.relative_to(PROJECT_ROOT)) if selected_path is not None else None,
            "artifact_type": metadata["artifact_type"],
            "status_role": metadata["status_role"],
        }
    )

artifact_inventory = pd.DataFrame(artifact_inventory_rows)
artifact_inventory

### Lightweight CSV schema validation

For CSV artifacts only, the next check validates whether the selected read-only artifact exists and, when present, reports table dimensions and expected-column coverage. Missing artifacts are reported without raising errors.

In [ ]:
EXPECTED_CSV_COLUMNS = {"sig_id", "core_score"}

csv_validation_rows = []
for artifact_name, metadata in ARTIFACT_REGISTRY.items():
    if metadata["artifact_type"] != "csv":
        continue

    preferred_path = metadata["preferred_path"]
    fallback_path = metadata["fallback_path"]
    selected_path = preferred_path if preferred_path is not None and preferred_path.exists() else fallback_path
    exists = selected_path.exists() if selected_path is not None else False

    row_count = None
    column_count = None
    missing_expected_columns = sorted(EXPECTED_CSV_COLUMNS)
    error = None

    if exists:
        try:
            csv_df = pd.read_csv(selected_path)
            row_count = len(csv_df)
            column_count = len(csv_df.columns)
            missing_expected_columns = sorted(EXPECTED_CSV_COLUMNS.difference(csv_df.columns))
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"

    csv_validation_rows.append(
        {
            "artifact": artifact_name,
            "exists": exists,
            "selected_path": str(selected_path.relative_to(PROJECT_ROOT)) if selected_path is not None else None,
            "row_count": row_count,
            "column_count": column_count,
            "missing_expected_columns": missing_expected_columns,
            "error": error,
        }
    )

csv_schema_validation = pd.DataFrame(csv_validation_rows)
csv_schema_validation

## 4B. Precomputed robustness correlation checks

These checks validate robustness using existing precomputed artifacts only. They do not recompute core scores, regenerate sensitivity outputs, create figures, or rewrite artifacts.

The read-only comparisons below use the artifact registry to compare precomputed scores for:

- `top30` vs `top50`
- `top100` vs `top50`
- `top30` vs `top100`


In [ ]:
ROBUSTNESS_SCORE_ARTIFACTS = ["top30_scores", "top50_scores", "top100_scores"]


def _path_label(path):
    if path is None:
        return None
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def _resolve_registered_artifact(artifact_name):
    metadata = ARTIFACT_REGISTRY.get(artifact_name)
    if metadata is None:
        return None, f"{artifact_name}: not present in ARTIFACT_REGISTRY"

    candidates = [
        ("preferred", metadata.get("preferred_path")),
        ("fallback", metadata.get("fallback_path")),
    ]
    checked = []
    for label, path in candidates:
        exists = path.exists() if path is not None else False
        checked.append(f"{label}={_path_label(path)} exists={exists}")
        if exists:
            return path, None

    return None, f"{artifact_name}: no readable artifact found ({'; '.join(checked)})"


robustness_score_paths = {}
robustness_loader_rows = []
for artifact_name in ROBUSTNESS_SCORE_ARTIFACTS:
    selected_path, warning = _resolve_registered_artifact(artifact_name)
    robustness_score_paths[artifact_name] = selected_path
    robustness_loader_rows.append(
        {
            "artifact": artifact_name,
            "selected_path": _path_label(selected_path),
            "available": selected_path is not None,
            "warning": warning,
        }
    )

robustness_loader_status = pd.DataFrame(robustness_loader_rows)
robustness_loader_status


In [ ]:
ROBUSTNESS_SCORE_COLUMNS = {
    "top30_scores": "score30",
    "top50_scores": "score50",
    "top100_scores": "score100",
}

robustness_score_tables_raw = {}
robustness_score_tables = {}
robustness_merge_messages = []

for artifact_name, score_column in ROBUSTNESS_SCORE_COLUMNS.items():
    selected_path = robustness_score_paths.get(artifact_name)
    if selected_path is None:
        robustness_merge_messages.append(f"{artifact_name}: skipped because no artifact path was resolved")
        continue

    try:
        raw_table = pd.read_csv(selected_path)
    except Exception as exc:
        robustness_merge_messages.append(f"{artifact_name}: could not read CSV ({type(exc).__name__}: {exc})")
        continue

    robustness_score_tables_raw[artifact_name] = raw_table
    required_columns = {"sig_id", "core_score"}
    missing_columns = sorted(required_columns.difference(raw_table.columns))
    if missing_columns:
        robustness_merge_messages.append(
            f"{artifact_name}: skipped because required columns are missing: {missing_columns}"
        )
        continue

    robustness_score_tables[artifact_name] = raw_table[["sig_id", "core_score"]].rename(
        columns={"core_score": score_column}
    )

required_tables_present = all(
    artifact_name in robustness_score_tables for artifact_name in ROBUSTNESS_SCORE_ARTIFACTS
)

if required_tables_present:
    robustness_scores_merged = (
        robustness_score_tables["top30_scores"]
        .merge(robustness_score_tables["top50_scores"], on="sig_id", how="inner")
        .merge(robustness_score_tables["top100_scores"], on="sig_id", how="inner")
    )
else:
    robustness_scores_merged = pd.DataFrame(columns=["sig_id", "score30", "score50", "score100"])

robustness_merge_validation = pd.DataFrame(
    [
        {
            "merged_row_count": len(robustness_scores_merged),
            "duplicate_sig_id_count": int(robustness_scores_merged["sig_id"].duplicated().sum())
            if "sig_id" in robustness_scores_merged.columns
            else None,
            "missing_score30": int(robustness_scores_merged["score30"].isna().sum())
            if "score30" in robustness_scores_merged.columns
            else None,
            "missing_score50": int(robustness_scores_merged["score50"].isna().sum())
            if "score50" in robustness_scores_merged.columns
            else None,
            "missing_score100": int(robustness_scores_merged["score100"].isna().sum())
            if "score100" in robustness_scores_merged.columns
            else None,
            "messages": robustness_merge_messages,
        }
    ]
)
robustness_merge_validation


In [ ]:
ROBUSTNESS_COMPARISONS = [
    ("top30 vs top50", "score30", "score50"),
    ("top100 vs top50", "score100", "score50"),
    ("top30 vs top100", "score30", "score100"),
]


def _score_correlations(frame, comparisons):
    rows = []
    for comparison_label, left_column, right_column in comparisons:
        if frame.empty or left_column not in frame.columns or right_column not in frame.columns:
            rows.append(
                {
                    "comparison": comparison_label,
                    "pearson_r": pd.NA,
                    "spearman_r": pd.NA,
                    "n": 0,
                }
            )
            continue

        paired_scores = frame[[left_column, right_column]].apply(pd.to_numeric, errors="coerce").dropna()
        rows.append(
            {
                "comparison": comparison_label,
                "pearson_r": paired_scores[left_column].corr(paired_scores[right_column], method="pearson")
                if len(paired_scores) >= 2
                else pd.NA,
                "spearman_r": paired_scores[left_column].corr(paired_scores[right_column], method="spearman")
                if len(paired_scores) >= 2
                else pd.NA,
                "n": len(paired_scores),
            }
        )
    return pd.DataFrame(rows, columns=["comparison", "pearson_r", "spearman_r", "n"])


global_robustness_correlations = _score_correlations(
    robustness_scores_merged,
    ROBUSTNESS_COMPARISONS,
)
global_robustness_correlations


### Per-cell-line robustness correlations

When `cell_id` is available in all three precomputed score tables, the next read-only check merges cell-line labels safely and calculates the same correlation summaries within each cell line. If any prerequisite is missing, the check reports a notebook-readable status and returns an empty summary table.


In [ ]:
per_cell_messages = []
per_cell_ready = required_tables_present

for artifact_name in ROBUSTNESS_SCORE_ARTIFACTS:
    raw_table = robustness_score_tables_raw.get(artifact_name)
    if raw_table is None:
        per_cell_ready = False
        per_cell_messages.append(f"{artifact_name}: raw score table is unavailable")
    elif "cell_id" not in raw_table.columns:
        per_cell_ready = False
        per_cell_messages.append(f"{artifact_name}: missing cell_id column")

if per_cell_ready:
    per_cell_inputs = []
    for artifact_name, score_column in ROBUSTNESS_SCORE_COLUMNS.items():
        cell_column = score_column.replace("score", "cell_id")
        per_cell_inputs.append(
            robustness_score_tables_raw[artifact_name][["sig_id", "cell_id", "core_score"]].rename(
                columns={"cell_id": cell_column, "core_score": score_column}
            )
        )

    per_cell_scores_merged = (
        per_cell_inputs[0]
        .merge(per_cell_inputs[1], on="sig_id", how="inner")
        .merge(per_cell_inputs[2], on="sig_id", how="inner")
    )
    cell_id_consistent = (
        (per_cell_scores_merged["cell_id30"] == per_cell_scores_merged["cell_id50"])
        & (per_cell_scores_merged["cell_id30"] == per_cell_scores_merged["cell_id100"])
    )
    mismatched_cell_id_count = int((~cell_id_consistent).sum())
    if mismatched_cell_id_count:
        per_cell_messages.append(
            f"Skipped {mismatched_cell_id_count} merged rows with inconsistent cell_id values across artifacts"
        )
    per_cell_scores_merged = per_cell_scores_merged.loc[cell_id_consistent].rename(
        columns={"cell_id30": "cell_id"}
    )

    per_cell_rows = []
    for cell_id, cell_frame in per_cell_scores_merged.groupby("cell_id", dropna=False):
        cell_summary = _score_correlations(cell_frame, ROBUSTNESS_COMPARISONS)
        cell_summary.insert(0, "cell_id", cell_id)
        per_cell_rows.append(cell_summary)

    per_cell_robustness_correlations = (
        pd.concat(per_cell_rows, ignore_index=True)
        if per_cell_rows
        else pd.DataFrame(columns=["cell_id", "comparison", "pearson_r", "spearman_r", "n"])
    )
else:
    per_cell_scores_merged = pd.DataFrame()
    mismatched_cell_id_count = None
    per_cell_robustness_correlations = pd.DataFrame(
        columns=["cell_id", "comparison", "pearson_r", "spearman_r", "n"]
    )

per_cell_robustness_status = pd.DataFrame(
    [
        {
            "per_cell_check_ran": per_cell_ready,
            "merged_row_count": len(per_cell_scores_merged),
            "mismatched_cell_id_count": mismatched_cell_id_count,
            "messages": per_cell_messages,
        }
    ]
)

per_cell_robustness_status, per_cell_robustness_correlations


### Validation scope

These checks validate already-generated score artifacts. They do not regenerate scores, they do not modify manuscript outputs, and they are intended to replace the safest read-only portions of `overlap_confirmation.ipynb`.


### Lightweight pickle validation

For pickle artifacts only, the next check confirms whether the selected read-only artifact exists and, when present, reports the loaded object type. Tuple artifacts also report tuple length and set sizes when the tuple contains sets. No pickle files are rewritten or regenerated.

In [ ]:
import pickle

pickle_validation_rows = []
for artifact_name, metadata in ARTIFACT_REGISTRY.items():
    if metadata["artifact_type"] != "pickle":
        continue

    preferred_path = metadata["preferred_path"]
    fallback_path = metadata["fallback_path"]
    selected_path = preferred_path if preferred_path is not None and preferred_path.exists() else fallback_path
    exists = selected_path.exists() if selected_path is not None else False

    object_type = None
    tuple_length = None
    set_sizes = None
    error = None

    if exists:
        try:
            with selected_path.open("rb") as handle:
                loaded_object = pickle.load(handle)
            object_type = type(loaded_object).__name__
            if isinstance(loaded_object, tuple):
                tuple_length = len(loaded_object)
                if all(isinstance(item, set) for item in loaded_object):
                    set_sizes = [len(item) for item in loaded_object]
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"

    pickle_validation_rows.append(
        {
            "artifact": artifact_name,
            "exists": exists,
            "selected_path": str(selected_path.relative_to(PROJECT_ROOT)) if selected_path is not None else None,
            "object_type": object_type,
            "tuple_length": tuple_length,
            "set_sizes": set_sizes,
            "error": error,
        }
    )

pickle_validation = pd.DataFrame(pickle_validation_rows)
pickle_validation

## Validation-only guarantees

- No scientific recomputation is performed.
- No sensitivity artifacts are regenerated.
- No manuscript outputs are modified.
- This notebook currently validates existing artifacts only.

## 5. Output policy

All future outputs from this notebook must be namespaced under `results/sensitivity/`.

This notebook must not write to canonical manuscript figure directories. Paper figures remain generated by `notebooks/04_directed_results.ipynb` only. The role of this notebook is to validate or regenerate sensitivity artifacts for robustness review, not to replace the canonical manuscript notebook.

## 6. TODO / future implementation steps

- Add read-only loaders for canonical and sensitivity core-score artifacts.
- Add variant-specific validation helpers that consume the `VARIANTS` registry.
- Add checks for expected schema, target counts, vote thresholds, and ranked-gene cutoffs.
- Add optional regeneration behind an explicit flag that defaults to disabled.
- Ensure regeneration writes only to `results/sensitivity/<variant>/`.
- Add a concise validation summary table for notebook reviewers.
- Keep manuscript figure generation and DOCX table generation out of this notebook.